# Telecom Customer Churn – Predictive Modelling

**Models:** Logistic Regression (baseline) vs XGBoost  
**Class imbalance handling:** `class_weight='balanced'` for LR; `scale_pos_weight` for XGBoost  
**Explainability:** SHAP global + individual explanations  
**Evaluation focus:** Recall — in churn prediction, missing a churner (false negative) is more costly than a false alarm.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load & Prepare Features

In [ ]:
from model_utils import load_and_prepare, CAT_FEATURES, NUM_FEATURES
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y, feature_cols = load_and_prepare()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Train churn rate: {y_train.mean():.1%}  |  Test churn rate: {y_test.mean():.1%}')
print(f'\nFeatures ({len(feature_cols)}): {feature_cols}')

## 2. Class Imbalance

The dataset has ~26% churn rate — moderately imbalanced. We handle this without oversampling (SMOTE) because:
- `class_weight='balanced'` in LR adjusts loss weights, keeping the training distribution real.
- `scale_pos_weight` in XGBoost does the same via the gradient weighting.
- Oversampling can introduce synthetic noise that hurts precision.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
y_train.value_counts().plot.bar(color=['#2196F3','#F44336'], ax=ax, edgecolor='white')
ax.set_xticklabels(['Retained (0)', 'Churned (1)'], rotation=0)
ax.set_title('Training Set Class Distribution')
ax.spines[['top','right']].set_visible(False)
for bar in ax.patches:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
            f'{int(bar.get_height()):,}', ha='center', fontsize=9)
plt.tight_layout(); plt.show()

## 3. Train Models

In [ ]:
from model_utils import train_models
results = train_models()
print('\nMetrics saved to models/model_metrics.csv')

## 4. Model Comparison

In [ ]:
metrics = results['metrics']
print(metrics.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(metrics))
width = 0.2
cols = ['recall', 'precision', 'f1', 'roc_auc']
colors = ['#4472C4', '#ED7D31', '#A9D18E', '#FFC000']
for i, (col, color) in enumerate(zip(cols, colors)):
    ax.bar([xi + i*width for xi in x], metrics[col], width=width,
           label=col.upper().replace('_','-'), color=color, edgecolor='white')
ax.set_xticks([xi + 1.5*width for xi in x])
ax.set_xticklabels(metrics['model'])
ax.set_ylim(0, 1.05)
ax.set_title('Model Performance Comparison')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../reports/figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Confusion Matrices

In [ ]:
from IPython.display import Image, display
display(Image('../reports/figures/cm_logistic_regression.png'))
display(Image('../reports/figures/cm_xgboost.png'))

## 6. ROC Curves

In [ ]:
display(Image('../reports/figures/roc_curves.png'))

## 7. SHAP – Global Feature Importance

SHAP (SHapley Additive exPlanations) attributes each prediction's deviation from the base rate to individual features.  
Red = higher feature value pushed prediction toward churn. Blue = lower value pushed toward retention.

In [ ]:
display(Image('../reports/figures/shap_summary.png'))

## 8. SHAP – Individual Explanation (Highest-Risk Customer)

In [ ]:
import shap
from model_utils import load_and_prepare, MODELS_PATH

xgb = joblib.load(Path('..') / 'models' / 'xgboost_churn.joblib')
X, y, feature_cols = load_and_prepare()
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)
xgb_probas = xgb.predict_proba(X_test)[:,1]
high_risk_idx = int(np.argmax(xgb_probas))

print(f'Highest-risk customer churn probability: {xgb_probas[high_risk_idx]:.1%}')
print('\nTop factors driving churn for this customer:')
top_shap = pd.Series(shap_values[high_risk_idx], index=feature_cols).abs().sort_values(ascending=False).head(5)
print(top_shap)

display(Image('../reports/figures/shap_waterfall_high_risk.png'))

## 9. Business Interpretation

| Finding | Business Action |
|---|---|
| Contract type is the strongest SHAP feature | Offer upgrade incentive to month-to-month customers at 6-month mark |
| Short tenure → high churn risk | Onboarding programme with satisfaction check at month 3 |
| High complaint tickets → high churn | Proactive outreach after 2nd complaint ticket |
| Fiber optic + electronic check → highest risk segment | Bundle discount + auto-pay setup offer |

The XGBoost model (AUC ~0.85) gives the retention team a reliable ranked list of at-risk customers each month — enabling targeted outreach before customers leave.